# S7a — measure a person from two phone photos, and pick the segmentation tool (Colab, GPU helps)

specs/s7a-photo-silhouette/spec.md (approved 25 Nov 2026). Two tools cut the person out of each photo
(A: torchvision, B: rembg; D-043). Photos stay on your Drive: nothing here commits, prints or saves them.

**Before you start:** put the test photos in `MyDrive/anthro/test_photos/` (front and side for each person,
ideally twice each, a minute apart). Use initials or numbers in file names, not full names.


## 1. Drive and data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/anthro"
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"
os.makedirs(DRIVE, exist_ok=True)

if os.path.exists(DATA_DIR):
    print("Data already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    !pip -q install awscli
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym

## 2. Get the code (with the optional photo tools)

In [ ]:
BRANCH = "s7a-photo-silhouette"   # set to "main" once merged
REPO = "https://github.com/Anthropometry/Anthropometry_Accademic_Model.git"
!rm -rf /content/repo && git clone -q --branch {BRANCH} {REPO} /content/repo
!pip -q install -e "/content/repo[photos]"
!git -C /content/repo log -1 --format="commit %h  %s"
GEO, CACHE = f"{DRIVE}/geo", f"{DRIVE}/cache"
RUN = f"{DRIVE}/runs/r4"
ANSUR = f"{DRIVE}/ansur2"
FORMULAS = f"{ANSUR}/ansur_formulas.json"

# A package installed in a running Colab session is only importable after a restart;
# pointing Python at the code directly avoids that.
import sys
sys.path.insert(0, "/content/repo/src")
import os
os.environ["TORCH_HOME"] = f"{DRIVE}/torch_cache"   # keep the downloaded tool weights between sessions
PHOTOS = f"{DRIVE}/test_photos"


## 3. Which way do BodyM's side silhouettes face? (seconds)
Side photos facing the other way are mirrored automatically (D-044). **Paste back:** the line printed.


In [ ]:
!anthro side-direction --data {DATA_DIR}


## 4. Your photos: fill in one line per photo pair
`height_cm` is the tape height. Tape measurements (optional, but they give the first real errors) go in
columns named by the app's codes, e.g. `waist_girth`, `hip_girth`, `bust_girth`. Use `None` when not measured.
The same `person` twice = two repeat pairs of that person.


In [ ]:
import pandas as pd
PAIRS = [
    # {"person": "P1", "front": "p1_front_1.jpg", "side": "p1_side_1.jpg", "height_cm": 174.0, "sex": "female",
    #  "age_range": "25-34", "weight_kg": 60.0, "waist_girth": None, "hip_girth": None},
]
df = pd.DataFrame(PAIRS)
for c in ("front", "side"):
    df[c] = PHOTOS + "/" + df[c]
df.to_csv(f"{PHOTOS}/pairs.csv", index=False)
print(len(df), "pairs,", df["person"].nunique(), "people")


## 5. Look at the outlines (both tools, first pair) — shown here only, never saved
Red line = where each tool thinks the body ends. Look at the waist, armpits and between the legs.


In [ ]:
from IPython.display import display
from PIL import Image
from anthro import photo, segment
first = pd.read_csv(f"{PHOTOS}/pairs.csv").iloc[0]
for name in ("torchvision", "rembg"):
    seg = segment.get(name)
    for view in ("front", "side"):
        rgb = photo.load_photo(first[view])
        body, _, _ = photo.clean(seg(rgb))
        print(name, view)
        display(Image.fromarray(photo.overlay(rgb, body)).resize((rgb.shape[1] // 3, rgb.shape[0] // 3)))


## 6. Compare the two tools (decided by the rule fixed in the spec)
**Paste back:** everything this cell prints (numbers only; no photos).


In [ ]:
!anthro compare-segmenters --pairs {PHOTOS}/pairs.csv --ckpt {RUN}/best.pt --data {DATA_DIR} --out {PHOTOS}/compare.json


## 7. One person's full result with the winning tool
Change `TOOL` to the winner printed above. **Paste back:** the result.


In [ ]:
TOOL = "rembg"
p = pd.read_csv(f"{PHOTOS}/pairs.csv").iloc[0]
F, S, H, SX = p.front, p.side, f"{p.height_cm:.1f}", p.sex
!anthro predict-photos --front "{F}" --side "{S}" --height {H} --sex {SX} --ckpt {RUN}/best.pt --tool {TOOL} --data {DATA_DIR}


## Before committing this notebook
**Edit → Clear all outputs** (it would contain your photos' outlines), then save. CI refuses notebooks with
saved outputs (POLICY 6).
